# **Proyek ETL Pipeline & Analisis Pemasaran Perbankan (Bank Marketing)**
## **Layer 2: Silver Layer (Data Cleansing, Deduplication & Feature Engineering)**

Notebook ini merupakan tahap kedua dari arsitektur Medallion. Pada tahap ini dilakukan:
1. **Membaca Data dari Bronze:** Mengambil data Parquet terstandardisasi dari `lakehouse/bronze/bank_marketing`.
2. **Data Cleansing & Deduplikasi:** Memastikan integritas data dan menghapus baris duplikat.
3. **Feature Engineering & Enrichment:** Menambahkan fitur turunan bisnis (seperti kategorisasi `AgeGroup`).
4. **Penyimpanan ke Silver Layer:** Menyimpan dataset yang bersih dan terstandarisasi ke Parquet (`lakehouse/silver/bank_marketing`) serta ekspor ke CSV tunggal.

---
### **1. Inisialisasi Sesi Spark**

In [ ]:
# Instal pyspark jika dijalankan di Google Colab terpisah
!pip install pyspark -q

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when

# Inisialisasi Spark Session
spark = SparkSession.builder \
    .appName("BankMarketing_02_Silver") \
    .getOrCreate()

print("Spark Session untuk Layer Silver berhasil diinisialisasi.")

---
### **2. Membaca Data dari Layer Bronze**

Memuat data mentah yang telah ditagging dari `lakehouse/bronze/bank_marketing`.

In [ ]:
bronze_path = "lakehouse/bronze/bank_marketing"

# Membaca data dari Layer Bronze
bronze_data = spark.read.parquet(bronze_path)
row_before = bronze_data.count()

print(f"Membaca data dari: {bronze_path}")
print(f"Jumlah baris awal dari Bronze: {row_before}")
bronze_data.printSchema()
bronze_data.show(5)

---
### **3. Data Cleansing & Deduplikasi**

Melakukan pembersihan data dengan menghapus baris duplikasi untuk menjamin kualitas data analitik.

In [ ]:
# Menghapus duplikasi data
silver_clean = bronze_data.dropDuplicates()

row_clean = silver_clean.count()
print(f"Jumlah baris sebelum pembersihan : {row_before}")
print(f"Jumlah baris setelah deduplikasi : {row_clean}")
print(f"Duplikasi yang dibersihkan       : {row_before - row_clean}")

---
### **4. Feature Engineering: Kategorisasi Kelompok Usia (`AgeGroup`)**

Menambahkan atribut baru yang membagi usia klien ke dalam 3 segmen:
* `Muda (<30)` : Usia di bawah 30 tahun.
* `Paruh Baya (30-50)` : Usia antara 30 hingga 50 tahun.
* `Tua (>50)` : Usia di atas 50 tahun.

In [ ]:
# Feature Engineering: Menambahkan Kolom AgeGroup
silver_final = silver_clean.withColumn(
    "AgeGroup",
    when(col("age") < 30, "Muda (<30)")
    .when((col("age") >= 30) & (col("age") <= 50), "Paruh Baya (30-50)")
    .otherwise("Tua (>50)")
)

print("Skema Data Silver setelah Feature Engineering:")
silver_final.printSchema()

# Tampilkan distribusi AgeGroup
print("\nDistribusi Kelompok Usia (AgeGroup):")
silver_final.groupBy("AgeGroup").count().show()

---
### **5. Penyimpanan ke Silver Layer & Ekspor Konsolidasi**

Menyimpan dataset yang sudah bersih dan diperkaya ke folder Parquet `lakehouse/silver/bank_marketing` dan file CSV tunggal di `lakehouse/exports/silver_export`.

In [ ]:
# Simpan ke Silver Layer (format Parquet)
silver_path = "lakehouse/silver/bank_marketing"
silver_final.write.mode("overwrite").parquet(silver_path)

row_after = silver_final.count()
print(f"Layer Silver Selesai.")
print(f"Lokasi penyimpanan: {silver_path}")
print(f"Jumlah baris tersimpan: {row_after}")

# Verifikasi membaca kembali data Silver
print("\n=== Sampel Data di Layer Silver ===")
spark.read.parquet(silver_path).select("age", "AgeGroup", "job", "balance", "duration", "y").show(5)

# Ekspor ke CSV Tunggal untuk downstream consumption
silver_export_path = "lakehouse/exports/silver_export"
silver_final.coalesce(1).write.mode("overwrite").option("header", "true").csv(silver_export_path)
print(f"Ekspor Silver ke CSV tunggal berhasil di: {silver_export_path}")

---
### **Ringkasan Layer Silver**
* Data telah bersih dari duplikasi dan terstandarisasi.
* Feature enrichment `AgeGroup` telah ditambahkan untuk mendukung segmentasi nasabah.
* Data siap digunakan oleh **Layer Gold** pada notebook berikutnya (`03_Gold_Layer.ipynb`) untuk pembuatan data mart analitik bisnis.